In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [3]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/KP/YASA_LOITERING DETECTION"

Mounted at /content/drive
/content/drive/MyDrive/KP/YASA_LOITERING DETECTION


In [4]:
import cv2
import torch
import time
from collections import defaultdict
from ultralytics import YOLO
import numpy as np

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
def detect_persons_with_time_tracking(model_path, video_path, output_path, detection_threshold_seconds=5):
    try:
        print(f"Loading YOLO model from: {model_path}")
        model = YOLO(model_path)
        print("YOLO model loaded successfully.")
    except Exception as e:
        print(f"Error loading YOLO model: {e}")
        return

    print(f"Opening video file: {video_path}")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video properties: Width={frame_width}, Height={frame_height}, FPS={fps}, Total Frames={total_frames}")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
    if not out.isOpened():
        print(f"Error: Could not create video writer for {output_path}")
        cap.release()
        return

    person_detection_start_times = {}
    person_last_bbox = {}

    frame_count = 0
    start_time = time.time()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        current_frame_time = time.time()

        results = model(frame, conf=0.5, classes=[0])

        current_frame_detected_persons = []

        for r in results:
            boxes = r.boxes
            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cls = int(box.cls[0])

                if model.names[cls] == 'human':
                    x_center = (x1 + x2) / 2
                    y_center = (y1 + y2) / 2

                    matched_person_id = None
                    min_distance = float('inf')
                    for person_id in person_last_bbox:
                        last_x_center, last_y_center = person_last_bbox[person_id]
                        distance = np.sqrt((x_center - last_x_center)**2 + (y_center - last_y_center)**2)
                        if distance < 50 and distance < min_distance:
                            min_distance = distance
                            matched_person_id = person_id

                    if matched_person_id is None:
                        person_id = (frame_count, x_center, y_center)
                        person_detection_start_times[person_id] = current_frame_time
                        person_last_bbox[person_id] = (x_center, y_center)
                        current_frame_detected_persons.append((person_id, (x1, y1, x2, y2)))
                    else:
                        person_last_bbox[matched_person_id] = (x_center, y_center)
                        current_frame_detected_persons.append((matched_person_id, (x1, y1, x2, y2)))

        for person_id, (x1, y1, x2, y2) in current_frame_detected_persons:
            detection_start_time = person_detection_start_times.get(person_id, current_frame_time)
            detection_duration = current_frame_time - detection_start_time

            color = (255, 0, 0)
            label = f"Person ({detection_duration:.1f}s)"

            if detection_duration > detection_threshold_seconds:
                color = (0, 0, 255)
                label = f"Loiterer ({detection_duration:.1f}s)"

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        out.write(frame)

    cap.release()
    out.release()

    end_time = time.time()
    print(f"Detection complete. Processed {frame_count} frames in {end_time - start_time:.2f} seconds.")
    print(f"Output video saved to: {output_path}")

In [15]:
if __name__ == "__main__":
    yolo_model_path = 'model.pt'
    input_video_path = 'sample_10.mp4'
    output_video_path = 'output_10.mp4'

    detect_persons_with_time_tracking(yolo_model_path, input_video_path, output_video_path, detection_threshold_seconds=3)

Loading YOLO model from: model.pt
YOLO model loaded successfully.
Opening video file: sample_10.mp4
Video properties: Width=1280, Height=720, FPS=24.0, Total Frames=192

0: 384x640 1 human, 11.3ms
Speed: 2.0ms preprocess, 11.3ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 human, 10.7ms
Speed: 1.7ms preprocess, 10.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 human, 11.2ms
Speed: 2.2ms preprocess, 11.2ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 human, 10.7ms
Speed: 2.5ms preprocess, 10.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 human, 10.7ms
Speed: 1.7ms preprocess, 10.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 human, 10.7ms
Speed: 2.4ms preprocess, 10.7ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 human, 10.7ms
Speed: 2.4ms preprocess, 10.7ms inference, 1.6ms post